# 📙 Bài 5 — SQL cơ bản

**Tuần 8**

> Rất nhiều dữ liệu doanh nghiệp nằm trong database, không phải file CSV. Biết SQL là điều kiện gần như bắt buộc khi đi phỏng vấn vị trí dữ liệu.

Dùng **SQLite** — có sẵn trong Python, không phải cài gì.

In [ ]:
import sqlite3
import pandas as pd
from pathlib import Path

DATA = Path("../../../data")
if not (DATA / "ban_hang.db").exists():
    DATA = Path("data")

con = sqlite3.connect(DATA / "ban_hang.db")


def sql(cau_lenh):
    """Chạy câu SQL và trả về DataFrame."""
    return pd.read_sql_query(cau_lenh, con)


sql("SELECT name FROM sqlite_master WHERE type='table'")

In [ ]:
sql("SELECT * FROM don_hang LIMIT 5")

## 1. SELECT — chọn cột và dòng

In [ ]:
sql("""
    SELECT don_id, danh_muc, so_luong, don_gia
    FROM don_hang
    WHERE don_gia > 30000000
    ORDER BY don_gia DESC
    LIMIT 5
""")

### ⭐ Thứ tự THỰC THI khác thứ tự VIẾT

```
Viết:      SELECT → FROM → WHERE → GROUP BY → HAVING → ORDER BY → LIMIT
Thực thi:  FROM → WHERE → GROUP BY → HAVING → SELECT → ORDER BY → LIMIT
             ①      ②        ③         ④        ⑤        ⑥        ⑦
```

Hiểu điều này giải thích rất nhiều lỗi khó hiểu:

| Câu hỏi | Trả lời |
|---|---|
| Vì sao không lọc được theo `SUM(...)` bằng `WHERE`? | `WHERE` chạy **trước** `GROUP BY` — lúc đó tổng chưa tồn tại. Phải dùng `HAVING` |
| Vì sao dùng được bí danh (alias) trong `ORDER BY` mà không dùng được trong `WHERE`? | `SELECT` (nơi đặt alias) chạy **sau** `WHERE` nhưng **trước** `ORDER BY` |

## 2. WHERE — lọc

In [ ]:
sql("""
    SELECT danh_muc, kenh, so_luong, don_gia
    FROM don_hang
    WHERE danh_muc IN ('Laptop', 'Dien thoai')
      AND kenh = 'Website'
      AND so_luong >= 2
      AND ngay_dat BETWEEN '2025-06-01' AND '2025-08-31'
    LIMIT 8
""")

| SQL | Ý nghĩa |
|---|---|
| `=` `!=` `>` `<` `>=` `<=` | So sánh |
| `AND` `OR` `NOT` | Nối điều kiện *(khác pandas — SQL dùng chữ)* |
| `IN (...)` | Thuộc danh sách *(tương đương `.isin()`)* |
| `BETWEEN a AND b` | Trong khoảng, **bao gồm** hai đầu |
| `LIKE 'Lap%'` | Khớp mẫu — `%` là nhiều ký tự, `_` là một ký tự |
| `IS NULL` / `IS NOT NULL` | Kiểm tra thiếu — **không dùng `= NULL`** |

> ⚠️ `NULL = NULL` cho kết quả `NULL` (không phải `TRUE`). Đây chính là bản sao của bẫy `nan != nan` trong NumPy. Luôn dùng `IS NULL`.

## 3. GROUP BY — tổng hợp

In [ ]:
sql("""
    SELECT
        danh_muc,
        COUNT(*)                        AS so_don,
        SUM(so_luong * don_gia)         AS doanh_thu,
        ROUND(AVG(don_gia))             AS gia_tb,
        MAX(don_gia)                    AS gia_cao_nhat
    FROM don_hang
    GROUP BY danh_muc
    ORDER BY doanh_thu DESC
""")

In [ ]:
# HAVING — lọc SAU khi tổng hợp
sql("""
    SELECT
        khach_id,
        COUNT(*)                  AS so_don,
        SUM(so_luong * don_gia)   AS tong_chi
    FROM don_hang
    GROUP BY khach_id
    HAVING so_don >= 12
    ORDER BY tong_chi DESC
    LIMIT 10
""")

> 🔑 **`WHERE` lọc DÒNG, `HAVING` lọc NHÓM.**
>
> ```sql
> WHERE  don_gia > 1000000     -- lọc từng đơn hàng, chạy TRƯỚC gom nhóm
> HAVING SUM(don_gia) > 5e9    -- lọc từng nhóm,      chạy SAU gom nhóm
> ```
>
> Hai câu này hoàn toàn khác nhau, và không thay thế cho nhau được.

**`COUNT(*)` vs `COUNT(cot)`:** `COUNT(*)` đếm mọi dòng; `COUNT(cot)` **bỏ qua** `NULL`. Đúng bằng khác biệt `size` vs `count` trong pandas.

## 4. JOIN — ghép bảng

In [ ]:
sql("""
    SELECT
        k.thanh_pho,
        COUNT(DISTINCT k.khach_id)   AS so_khach,
        COUNT(d.don_id)              AS so_don,
        SUM(d.so_luong * d.don_gia)  AS doanh_thu
    FROM don_hang AS d
    JOIN khach_hang AS k ON d.khach_id = k.khach_id
    GROUP BY k.thanh_pho
    ORDER BY doanh_thu DESC
""")

> Kết quả cho thấy cột `thanh_pho` **chưa được chuẩn hoá** — `"HN"`, `"ha noi"`, `"Hà Nội"` bị tính thành các nhóm riêng. Trong công việc thật, việc chuẩn hoá thường phải làm ở tầng dữ liệu trước khi báo cáo, chứ không phải sửa từng câu truy vấn.

| JOIN | Giữ lại gì | Tương đương pandas |
|---|---|---|
| `INNER JOIN` (mặc định) | Chỉ dòng khớp cả hai | `how="inner"` |
| `LEFT JOIN` | Hết bảng trái + phần khớp | `how="left"` |
| `RIGHT` / `FULL OUTER` | (SQLite bản cũ không hỗ trợ) | `how="right"` / `"outer"` |

> ⚠️ Đúng như trong pandas: nếu bảng bên phải có khoá **trùng lặp**, `JOIN` sẽ **nhân bản** các dòng bên trái. Luôn kiểm tra số dòng trước và sau.

In [ ]:
# LEFT JOIN — tìm khách chưa từng mua hàng
sql("""
    SELECT COUNT(*) AS khach_chua_mua
    FROM khach_hang AS k
    LEFT JOIN don_hang AS d ON k.khach_id = d.khach_id
    WHERE d.don_id IS NULL
""")

## 5. Truy vấn con & CTE

In [ ]:
# CTE (WITH) — đặt tên cho kết quả trung gian, dễ đọc hơn truy vấn lồng
sql("""
    WITH chi_tieu AS (
        SELECT khach_id, SUM(so_luong * don_gia) AS tong_chi
        FROM don_hang
        GROUP BY khach_id
    )
    SELECT
        CASE
            WHEN tong_chi >= 100000000 THEN 'VIP'
            WHEN tong_chi >=  50000000 THEN 'Thân thiết'
            ELSE 'Thường'
        END                AS phan_khuc,
        COUNT(*)           AS so_khach,
        ROUND(AVG(tong_chi)) AS chi_tieu_tb
    FROM chi_tieu
    GROUP BY phan_khuc
    ORDER BY chi_tieu_tb DESC
""")

**CTE (`WITH ... AS`)** cho phép đặt tên cho một bước trung gian. Với truy vấn phức tạp, nó là khác biệt giữa code đọc được và code không ai dám sửa.

**`CASE WHEN`** là `if/elif/else` của SQL — tương đương `pd.cut()` hoặc `np.select()`.

## 6. Window function — công cụ mạnh nhất của SQL

In [ ]:
sql("""
    WITH theo_thang AS (
        SELECT
            substr(ngay_dat, 1, 7)       AS thang,
            SUM(so_luong * don_gia)      AS doanh_thu
        FROM don_hang
        WHERE ngay_dat LIKE '2025-%'
        GROUP BY thang
    )
    SELECT
        thang,
        doanh_thu,
        LAG(doanh_thu) OVER (ORDER BY thang)  AS thang_truoc,
        ROUND(
            100.0 * (doanh_thu - LAG(doanh_thu) OVER (ORDER BY thang))
            / LAG(doanh_thu) OVER (ORDER BY thang), 1
        )                                     AS tang_truong_pt
    FROM theo_thang
    ORDER BY thang
""")

**Window function** tính toán trên một "cửa sổ" các dòng **mà không gom nhóm lại** — chính là `transform` của pandas.

| Hàm | Tác dụng |
|---|---|
| `LAG(x)` / `LEAD(x)` | Giá trị dòng trước / dòng sau — dùng tính tăng trưởng |
| `ROW_NUMBER()` | Đánh số thứ tự |
| `RANK()` / `DENSE_RANK()` | Xếp hạng (có/không nhảy số khi đồng hạng) |
| `SUM(x) OVER (...)` | Tổng luỹ kế |

Đây là chủ đề **rất hay hỏi phỏng vấn** cho vị trí data.

In [ ]:
# Top 2 đơn lớn nhất TRONG MỖI danh mục
sql("""
    WITH xep_hang AS (
        SELECT
            danh_muc, don_id, so_luong * don_gia AS gia_tri,
            ROW_NUMBER() OVER (
                PARTITION BY danh_muc ORDER BY so_luong * don_gia DESC
            ) AS hang
        FROM don_hang
    )
    SELECT danh_muc, don_id, gia_tri
    FROM xep_hang
    WHERE hang <= 2
    ORDER BY danh_muc, hang
""")

`PARTITION BY` = chia thành các nhóm rồi đánh số **trong từng nhóm**. Bài toán "top N mỗi nhóm" này xuất hiện liên tục trong công việc thật.

## 7. SQL ↔ Pandas

| SQL | Pandas |
|---|---|
| `SELECT a, b` | `df[["a", "b"]]` |
| `WHERE x > 5` | `df[df["x"] > 5]` |
| `WHERE x IN (...)` | `df[df["x"].isin([...])]` |
| `ORDER BY x DESC` | `df.sort_values("x", ascending=False)` |
| `LIMIT 10` | `df.head(10)` |
| `GROUP BY a` | `df.groupby("a")` |
| `COUNT(*)` | `.size()` |
| `COUNT(col)` | `.count()` |
| `HAVING n > 5` | `.query("n > 5")` sau khi agg |
| `JOIN ... ON` | `pd.merge(..., on=...)` |
| `LEFT JOIN` | `pd.merge(..., how="left")` |
| `CASE WHEN` | `np.select()` / `pd.cut()` |
| `LAG(x) OVER (...)` | `df.groupby(...)["x"].shift(1)` |
| `SUM(x) OVER (...)` | `df.groupby(...)["x"].transform("sum")` |

**Khi nào dùng cái nào?**
- **SQL** khi dữ liệu nằm trong database và **lớn** — lọc/tổng hợp ngay tại DB, chỉ kéo về phần cần dùng
- **Pandas** khi dữ liệu đã ở trong bộ nhớ, hoặc cần biến đổi phức tạp, vẽ biểu đồ, chạy model

**Quy trình thực tế phổ biến nhất:** SQL để lấy + thu gọn dữ liệu → pandas để phân tích sâu → matplotlib để trình bày.

In [ ]:
# Kết hợp: SQL lấy dữ liệu, pandas phân tích tiếp
bang = sql("""
    SELECT k.thanh_pho, d.danh_muc, SUM(d.so_luong * d.don_gia) AS doanh_thu
    FROM don_hang d
    JOIN khach_hang k ON d.khach_id = k.khach_id
    GROUP BY k.thanh_pho, d.danh_muc
""")

(bang.pivot_table(index="thanh_pho", columns="danh_muc",
                  values="doanh_thu", aggfunc="sum")
     .fillna(0)
     .astype(int)
     .head())

In [ ]:
con.close()
print("Đã đóng kết nối database.")

> 📌 **Luôn đóng kết nối.** Hoặc dùng context manager để tự đóng:
> ```python
> with sqlite3.connect("ban_hang.db") as con:
>     df = pd.read_sql_query("SELECT * FROM don_hang", con)
> ```

## ✍️ Bài tập tự làm

Không có bài chấm tự động cho SQL. Hãy tự viết các truy vấn trả lời:

1. Tháng nào doanh thu cao nhất?
2. Khách hàng nào mua nhiều danh mục khác nhau nhất?
3. Kênh nào có giá trị đơn trung bình cao nhất?
4. Tỷ lệ đơn có giảm giá là bao nhiêu?
5. Top 3 khách chi tiêu nhiều nhất **ở mỗi thành phố** *(gợi ý: window function)*

Kiểm chứng: làm lại **cùng câu hỏi đó bằng pandas** và so sánh kết quả. Nếu hai bên ra số khác nhau, bạn vừa tìm được một hiểu lầm đáng giá.

---

🎯 **Xong Tuần 8 → làm [project P1](../../../projects/P1-eda-report/README.md)**